<a href="https://colab.research.google.com/github/pcmay/ALyzer3D.AI/blob/main/ALyzer3DAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



<div style="display: flex; justify-content: space-between; align-items: center;">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/white.png" width="10%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/ALyzer3D.AI_logo.png" width="25%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/white.png" width="25%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/ColabFold_logo.png" width="25%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/white.png" width="10%">
</div>



Welcome to **ALyzer3D.AI**. This notebook allows you to predict the amyloidogenicity of a VL domain sequence of a light chain by first generating its 3D structure with [ColabFold](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb) and then automatically analyzing it with the ALyzer3D.AI model.

**Instructions:**

1. **Enter Your Sequence**: In the first cell (query_sequence), paste the amino acid sequence of your light chain's VL domain.
2. **Select a GPU**: Click Runtime, select Change runtime type, select T4 GPU (or any GPU option available - DO NOT use TPUs). Click Save.
3. **Run Everything**: Click on the menu Runtime -> Run all.

The notebook will now execute all the steps for you: it will install dependencies, run the ColabFold structure prediction (ca. 5 min), and finally, perform the ALyzer3D.AI analysis on the resulting top-ranked structure. The final prediction will be displayed at the bottom of the page.



In [ ]:
#@title Run ColabFold v1.5.5 Prediction

#@markdown ### Enter your protein sequence
query_sequence = 'DIRLTQSPSSLSASVGDRVTITCQASQHINNYLNWYQHKPGQAPKVLIYDASNLATGVPSRFSGNGSGTHFTLTINSLQPEDAATYYCQQHDDLPLTFGGGTKVEIT' #@param {type:"string"}


# --- Standard Parameters (do not change unless you know what you are doing) ---
jobname = 'colabfold_prediction'
num_relax = 0
template_mode = "none"
msa_mode = "mmseqs2_uniref_env"
model_type = "auto"
pair_mode = "unpaired_paired"
num_recycles = 3
# -----------------------------------------------------------------------------

import os
import sys
from sys import version_info

# Check if ColabFold and its dependencies are already installed
if not os.path.isfile("COLABFOLD_READY"):
    print("Installing ColabFold...")
    # Install ColabFold
    os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")

    # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    # !! THIS IS THE FIX for the TensorFlow "undefined symbol" error !!
    # !! We remove the problematic library file that causes the crash. !!
    # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so")

    # ❌ REMOVED: The line below was breaking TensorFlow Lite!
    # os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/lite/python/*/*.so")

    # Create symbolic links for easier access
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
    # Mark installation as complete
    os.system("touch COLABFOLD_READY")

# --- Code Execution Starts Here ---
import re
import hashlib
from pathlib import Path
import warnings
from Bio import BiopythonDeprecationWarning

# Suppress unnecessary warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=BiopythonDeprecationWarning)

from colabfold.download import download_alphafold_params
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type

# Define a function to create a unique jobname
def get_unique_jobname(basename, sequence):
    # Create a hash from the sequence to make the jobname unique
    sequence_hash = hashlib.sha1(sequence.encode()).hexdigest()[:5]
    unique_name = f"{basename}_{sequence_hash}"

    # Handle potential jobname collisions
    if os.path.exists(unique_name):
        n = 0
        while os.path.exists(f"{unique_name}_{n}"):
            n += 1
        unique_name = f"{unique_name}_{n}"
    return unique_name

# Sanitize and prepare inputs
query_sequence = "".join(query_sequence.split())
sanitized_jobname = re.sub(r'\W+', '', jobname)
jobname = get_unique_jobname(sanitized_jobname, query_sequence)

# Create directory for results
os.makedirs(jobname, exist_ok=True)

# Write sequence to a CSV file for ColabFold
queries_path = os.path.join(jobname, f"{jobname}.csv")
with open(queries_path, "w") as text_file:
    text_file.write(f"id,sequence\n{jobname},{query_sequence}")

print(f"Starting prediction for job: {jobname}")
print(f"Sequence length: {len(query_sequence.replace(':', ''))}")

# Set up logging
result_dir = Path(jobname)
setup_logging(result_dir.joinpath("log.txt"))

# Parse inputs for the ColabFold `run` function
queries, is_complex = get_queries(queries_path)
model_type = set_model_type(is_complex, model_type)
num_recycles_parsed = None if num_recycles == "auto" else int(num_recycles)
use_templates = template_mode != "none"

# Download AlphaFold parameters
print("Downloading model parameters...")
download_alphafold_params(model_type, Path("."))

# Execute the main prediction function
print("Running prediction...")
run(
    queries=queries,
    result_dir=result_dir,
    use_templates=use_templates,
    custom_template_path=None,
    num_relax=num_relax,
    msa_mode=msa_mode,
    model_type=model_type,
    num_models=5,  # Standard number of models
    num_recycles=3,
    num_seeds=1,
    model_order=[1, 2, 3, 4, 5],
    is_complex=is_complex,
    data_dir=Path("."),
    keep_existing_results=False,
    rank_by="auto",
    pair_mode=pair_mode,
    stop_at_score=100.0,
    zip_results=False, # We will zip manually at the end
    user_agent="colabfold/google-colab-main",
)

# Package results into a zip file
print("Packaging results...")
results_zip_path = f"{jobname}.result.zip"
os.system(f"zip -r -q {results_zip_path} {jobname}")

print(f"\nDone! Prediction complete.")
print(f"Results are saved in '{results_zip_path}'.")

In [ ]:
#@title ▶️ Run ALyzer3D.AI Analysis

VERBOSE = False  #@param {type:"boolean"}

# The notebook kernel hosts JAX/ColabFold, so the analysis runs in a separate
# CPU-only interpreter. TF/Keras are never imported here.

import os, sys, glob, json, subprocess
from IPython.display import display, HTML

REPO_PATH = "/content/ALyzer3D.AI"
MODEL_FOLDER_NAME = "paper_model_scalar_pathway_v1_minus5_stripped_80_20_seed3"
SCRIPT_PATH = "/content/run_alyzer.py"

def log(msg):
    if VERBOSE:
        print(msg)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers", "scikit-learn", "joblib", "biopython", "pandas"],
    check=True, capture_output=True,
)
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", "https://github.com/petercmay89/ALyzer3D.AI.git",
                    REPO_PATH], check=True, capture_output=True)

# ------------------------------------------------------------------------------
# Locate ColabFold output. Do not rely on `jobname` from cell 1: a kernel restart
# wipes it while the results stay on disk.
# ------------------------------------------------------------------------------
if "jobname" not in globals() or not os.path.isdir(globals().get("jobname", "")):
    candidates = sorted(
        (d for d in glob.glob("*") if os.path.isdir(d)
         and glob.glob(f"{d}/{d}_unrelaxed_rank_001*.pdb")),
        key=os.path.getmtime,
    )
    if not candidates:
        raise FileNotFoundError("No ColabFold result folder found. Run cell 1.")
    jobname = candidates[-1]
    print(f"Recovered job from disk: {jobname}")
    if len(candidates) > 1:
        print(f"  (most recent of {len(candidates)}: {candidates})")

pdb_files  = sorted(glob.glob(f"{jobname}/{jobname}_unrelaxed_rank_001*.pdb"))
json_files = sorted(glob.glob(f"{jobname}/{jobname}_scores_rank_001*.json"))
if not pdb_files or not json_files:
    raise FileNotFoundError(f"No rank_001 PDB/JSON in '{jobname}'. Re-run cell 1.")
pdb_filename, json_filename = pdb_files[0], json_files[0]
log(f" - PDB:  {pdb_filename}")
log(f" - JSON: {json_filename}")

# ------------------------------------------------------------------------------
# Worker script
# ------------------------------------------------------------------------------
WORKER = r'''
import os, sys, re, glob, json
import numpy as np, joblib, torch, keras

MODEL_DIR, PDB_PATH, JSON_PATH, OUT_PATH, VERBOSE = sys.argv[1:6]
VERBOSE = VERBOSE == "1"
MAX_LENGTH = 120
THRESHOLD = 0.60          # Youden-optimised for seed 3 (paper, Table 2)
PLM_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

def log(msg):
    if VERBOSE:
        print(f"[worker] {msg}", flush=True)

if not keras.__version__.startswith("3"):
    sys.exit(f"Keras 3 required, found {keras.__version__}")
log(f"keras {keras.__version__}")

from transformers import AutoTokenizer, EsmModel
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.PDBData import protein_letters_3to1
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# --- Lambda compat: 'normalize_length' is stored as marshalled bytecode from a
# --- different Python version. marshal.loads() on that can segfault the
# --- interpreter outright, so intercept BEFORE deserialization is attempted.
# --- A try/except cannot help here: a segfault is not an exception.
_ORIG = keras.layers.Lambda.from_config.__func__
_BYPASSED = []

def _safe(cls, config, *a, **kw):
    fn = config.get("function")
    if isinstance(fn, dict) and fn.get("class_name") == "__lambda__":
        closure = fn.get("config", {}).get("closure") or []
        divisor = float(closure[0]) if closure else float(MAX_LENGTH)
        _BYPASSED.append(divisor)
        log(f"bypassing marshalled Lambda '{config.get('name')}' (x / {divisor})")

        def _normalize_length(x, _d=divisor):
            return x / _d

        return cls(function=_normalize_length,
                   name=config.get("name"),
                   trainable=config.get("trainable", True))
    return _ORIG(cls, config, *a, **kw)

keras.layers.Lambda.from_config = classmethod(_safe)

def fold_id(p):
    m = re.search(r"fold[_-]?(\d+)", os.path.basename(p), re.IGNORECASE)
    return int(m.group(1)) if m else None

def pair_files(models, scalers):
    m = {fold_id(p): p for p in models}
    s = {fold_id(p): p for p in scalers}
    if None in m or None in s or len(m) != len(models) or len(s) != len(scalers):
        # Always printed: silent mispairing of models and scalers is a data bug.
        print("[worker] WARNING: fold numbers unreadable, using sorted pairing", flush=True)
        return list(zip(sorted(models), sorted(scalers)))
    missing = (set(m) | set(s)) - (set(m) & set(s))
    if missing:
        sys.exit(f"Folds without a matching pair: {sorted(missing)}")
    return [(m[i], s[i]) for i in sorted(set(m) & set(s))]

log("loading ESM-2...")
device = torch.device("cpu")
tokenizer = AutoTokenizer.from_pretrained(PLM_MODEL_NAME)
plm = EsmModel.from_pretrained(PLM_MODEL_NAME).to(device).eval()

log("loading ensemble...")
model_files = glob.glob(os.path.join(MODEL_DIR, "*.keras")) or \
              glob.glob(os.path.join(MODEL_DIR, "*.h5"))
scaler_files = glob.glob(os.path.join(MODEL_DIR, "*.joblib"))
if not model_files or not scaler_files:
    sys.exit(f"{len(model_files)} models / {len(scaler_files)} scalers in {MODEL_DIR}")

models, scalers = [], []
for mp, sp in pair_files(model_files, scaler_files):
    log(f"{os.path.basename(mp)} <-> {os.path.basename(sp)}")
    # safe_mode stays at its default (True) on purpose: if any Lambda slips past
    # the patch above, Keras raises a readable ValueError instead of segfaulting.
    models.append(keras.models.load_model(mp, compile=False))
    scalers.append(joblib.load(sp))

# Divisors must agree across folds; disagreement means inconsistent training.
if len(set(_BYPASSED)) > 1:
    sys.exit(f"Folds disagree on length normalisation: {sorted(set(_BYPASSED))}")
log(f"{len(models)} fold(s) loaded")

# --- Features. No bare excepts: a silent fallback feature vector would produce a
# --- confident-looking probability from garbage input.
parser = PDBParser(QUIET=True)
structure = parser.get_structure("s", PDB_PATH)[0]

chain = next(structure.get_chains())
sequence = "".join(protein_letters_3to1.get(r.get_resname().upper(), "X")
                   for r in chain.get_residues() if is_aa(r, standard=True))
if not sequence:
    sys.exit(f"No standard residues parsed from {PDB_PATH}")

atoms = list(structure.get_atoms())
n_res = len(list(structure.get_residues()))
if not atoms or n_res == 0:
    sys.exit(f"No atoms/residues parsed from {PDB_PATH}")
com = sum(a.coord for a in atoms) / len(atoms)
rog = float(np.sqrt(sum(np.sum((a.coord - com) ** 2) for a in atoms) / len(atoms))
            / np.sqrt(n_res))

seq_clean = "".join(c for c in sequence if c in "ACDEFGHIKLMNPQRSTVWY")
pa = ProteinAnalysis(seq_clean)
biochem = [pa.isoelectric_point(), pa.gravy(), pa.aromaticity(), pa.molecular_weight()]

with open(JSON_PATH) as f:
    data = json.load(f)
plddt = np.array(data["plddt"], dtype="float32")
pae = np.array(data["pae"], dtype="float32")

L = min(len(sequence), len(plddt), pae.shape[0])
effective_len = L - 5
if effective_len <= 0:
    sys.exit(f"Protein too short (len={L}) for -5 truncation.")
n = min(effective_len, MAX_LENGTH)

pad_pae = np.zeros((MAX_LENGTH, MAX_LENGTH), dtype="float32")
pad_pae[:n, :n] = pae[:n, :n]
pad_plddt = np.zeros(MAX_LENGTH, dtype="float32"); pad_plddt[:n] = plddt[:n]
pad_row = np.zeros(MAX_LENGTH, dtype="float32")
pad_col = np.zeros(MAX_LENGTH, dtype="float32")
pad_row[:n] = np.mean(pae[:n, :n], axis=1)
pad_col[:n] = np.mean(pae[:n, :n], axis=0)

with torch.no_grad():
    tok = tokenizer(sequence, return_tensors="pt", truncation=True, max_length=1022)
    emb = plm(**tok).last_hidden_state.squeeze(0).mean(dim=0).cpu().numpy()

raw_scalars = np.array(biochem + [rog], dtype="float32").reshape(1, -1)
base = {
    "pae_input":       np.expand_dims(pad_pae, [0, -1]),
    "plddt_input":     np.expand_dims(pad_plddt, [0, -1]),
    "embedding_input": np.expand_dims(emb, 0).astype("float32"),
    "pae_row_input":   np.expand_dims(pad_row, [0, -1]),
    "pae_col_input":   np.expand_dims(pad_col, [0, -1]),
    "length_input":    np.array([[effective_len]], dtype="float32"),  # (1,1), not (1,)
}

fold_preds = []
for mdl, sc in zip(models, scalers):
    inp = dict(base)
    inp["scalar_features_input"] = sc.transform(raw_scalars).astype("float32")
    fold_preds.append(float(mdl.predict(inp, verbose=0)[0][0]))

avg = float(np.mean(fold_preds))
with open(OUT_PATH, "w") as f:
    json.dump({"sequence": sequence, "probability": avg,
               "label": "AMYLOID" if avg > THRESHOLD else "NON-AMYLOID",
               "threshold": THRESHOLD,
               "fold_scores": fold_preds, "rog": rog, "biochem": biochem,
               "effective_len": int(effective_len),
               "mean_plddt": float(np.mean(plddt[:L]))}, f)
log("done")
'''

with open(SCRIPT_PATH, "w") as f:
    f.write(WORKER)

# ------------------------------------------------------------------------------
# Run it
# ------------------------------------------------------------------------------
print("Running analysis...")
out_path = os.path.join(jobname, "_alyzer_raw.json")
env = dict(os.environ)
env["CUDA_VISIBLE_DEVICES"] = "-1"      # keep TF off the GPU entirely
env.pop("TF_USE_LEGACY_KERAS", None)    # models are Keras 3 format
env["TF_CPP_MIN_LOG_LEVEL"] = "3"
env["TRANSFORMERS_VERBOSITY"] = "error"
env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"

proc = subprocess.run(
    [sys.executable, SCRIPT_PATH,
     os.path.join(REPO_PATH, MODEL_FOLDER_NAME),
     pdb_filename, json_filename, out_path, "1" if VERBOSE else "0"],
    env=env, capture_output=True, text=True,
)

# Warnings from the worker are surfaced even when quiet.
for line in proc.stdout.splitlines():
    if VERBOSE or "WARNING" in line:
        print(line)

if proc.returncode != 0:
    print(proc.stderr[-4000:])
    if proc.returncode == -11:
        raise RuntimeError(
            "Worker segfaulted (SIGSEGV). Set VERBOSE = True and re-run to see "
            "how far it got before dying."
        )
    if proc.returncode < 0:
        raise RuntimeError(f"Worker killed by signal {-proc.returncode}.")
    raise RuntimeError(f"Worker exited with code {proc.returncode}")

with open(out_path) as f:
    result = json.load(f)

# ------------------------------------------------------------------------------
# Report
# ------------------------------------------------------------------------------
prob, label = result["probability"], result["label"]
confidence_percent = prob * 100
folds = result["fold_scores"]

import pandas as pd
results_csv_path = os.path.join(jobname, f"{jobname}_alyzer_results.csv")
pd.DataFrame([{
    "ID": jobname, "Prediction": label, "Probability": prob,
    "Confidence_Percent": round(confidence_percent, 2),
    "Threshold": result["threshold"],
    "N_Folds": len(folds),
    "Fold_Min": min(folds), "Fold_Max": max(folds),
    "Fold_Scores": ";".join(f"{p:.6f}" for p in folds),
    "Mean_pLDDT": result["mean_plddt"],
    "RoG": result["rog"], "Effective_Len": result["effective_len"],
    "Sequence": result["sequence"],
}]).to_csv(results_csv_path, index=False)

zip_file_name = f"{jobname}.result.zip"
if os.path.exists(zip_file_name):
    subprocess.run(["zip", "-u", "-q", zip_file_name, results_csv_path])

log(f" - fold scores: {[round(p, 4) for p in folds]}")
log(f" - RoG {result['rog']:.4f} | effective_len {result['effective_len']}"
    f" | mean pLDDT {result['mean_plddt']:.1f}")
log(f" - CSV: {results_csv_path}")

# Single source of truth: the colour follows the worker's label, it does not
# re-threshold. Otherwise heading and 'Prediction' field can contradict.
risk_level, risk_color = (("High Risk (Amyloid)", "#D32F2F")
                          if label == "AMYLOID"
                          else ("Low Risk (Non-Amyloid)", "#388E3C"))

html_output = f"""
<div style="border: 2px solid {risk_color}; border-radius: 10px; padding: 20px; font-family: sans-serif; background-color: #f9f9f9; margin-top: 1em;">
    <h2 style="color: {risk_color}; margin-top: 0;">ANALYSIS COMPLETE: {risk_level.upper()}</h2>
    <hr>
    <div style="display: grid; grid-template-columns: 150px 1fr; gap: 10px; align-items: center;">

        <strong style="font-size: 1.1em;">Prediction:</strong>
        <span style="font-size: 1.1em; font-weight: bold; color: {risk_color};">{label}</span>

        <strong style="font-size: 1.1em;">Probability:</strong>
        <div style="width: 100%; background-color: #e0e0e0; border-radius: 5px;">
            <div style="width: {confidence_percent}%; background-color: {risk_color}; color: white; text-align: center; padding: 2px 0; border-radius: 5px; min-width: 30px;">
                {confidence_percent:.2f}%
            </div>
        </div>

        <strong style="vertical-align: top;">Sequence:</strong>
        <textarea readonly style="width: 100%; height: 60px; resize: none; border: 1px solid #ccc; font-family: monospace; background-color: #fff;">{result['sequence']}</textarea>
    </div>
</div>
"""
display(HTML(html_output))

In [ ]:
#@title Download Results
from google.colab import files

# Download the zip file created in the prediction cell
files.download(f"{jobname}.result.zip")

# Instructions <a name="Instructions"></a>
For detailed instructions, tips and tricks on ColabFold, see recently published paper at [Nature Protocols](https://www.nature.com/articles/s41596-024-01060-5)